# Lab 02-1 - Hierarchical Clustering

In this lab we will transition towards focusing on machine learning methods themselves and away from the inner working of Python, `numpy`, and `pandas`. You should be prepared to use information from previous labs, but also to use published documentation to learn about useful functions, methods, and attributes on your own.

In [ ]:
## Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

Examples throughout the lab will use the "moons" and "blobs" toy data sets that were introduced in a previous lab:

In [ ]:
## Datasets module
from sklearn import datasets

## Toy Dataset #1 - "moons"
moons = datasets.make_moons(n_samples=500, noise=0.11, random_state=8)
moons_X = moons[0]         # features to use in clustering
moons_labels = moons[1]    # true cluster identities

## Visualization
plt.scatter(moons_X[:,0], moons_X[:,1], c = moons_labels)
plt.show()

## Toy Dataset #2 - "blobs"
blobs = datasets.make_blobs(n_samples=500, cluster_std=2, random_state=8)
blobs_X = blobs[0]         # features to use in clustering
blobs_labels = blobs[1]    # true cluster identities

## Visualization
plt.scatter(blobs_X[:,0], blobs_X[:,1], c = blobs_labels)
plt.show()

## Part 1 - Agglomerative clustering

### Model Fitting and Dendrograms

Agglomerative clustering is implemented in the `cluster` module of `sklearn`. The basic workflow of many models implemented in sklearn is to first initialize a model then fit to your data using its `.fit()` method below. The arguments `distance_threshold=0` and `n_clusters=None` are used to ensure that the full dendrogram is fit to the data.

In [ ]:
from sklearn.cluster import AgglomerativeClustering
agg_cluster_single = AgglomerativeClustering(
    linkage = 'single', 
    distance_threshold=0, 
    n_clusters=None).fit(moons_X)

Unfortunately there not currently an official function to plot dendrograms in `sklearn`, but the following function, which is [described here](https://scikit-learn.org/stable/auto_examples/cluster/plot_agglomerative_dendrogram.html), can be used to create a basic dendrogram. Don't concern yourself too much with understanding how this code works; rather take it as given.

In [ ]:
from scipy.cluster.hierarchy import dendrogram

def plot_dendrogram(model, **kwargs):
    # Create linkage matrix and then plot the dendrogram

    # create the counts of samples under each node
    counts = np.zeros(model.children_.shape[0])
    n_samples = len(model.labels_)
    for i, merge in enumerate(model.children_):
        current_count = 0
        for child_idx in merge:
            if child_idx < n_samples:
                current_count += 1  # leaf node
            else:
                current_count += counts[child_idx - n_samples]
        counts[i] = current_count

    linkage_matrix = np.column_stack(
        [model.children_, model.distances_, counts]
    ).astype(float)

    # Plot the corresponding dendrogram
    dendrogram(linkage_matrix, **kwargs)

In [ ]:
## Part of the dendrogram for "moons"

# The p argument sets how deep down the dendrogram tree should we plot.
# Try different values and see how the numbers in parentheses change
plot_dendrogram(model = agg_cluster_single, truncate_mode="level", p=10) 
plt.xlabel("Index of obs (or # of obs in branch if in parentheses)")
plt.ylabel("Distance between merged clusters")
plt.show()

A few things you should notice in this example:

1. The function relies upon the `dendogram()` function in the `scipy` library. This function is assigning colors to the branches of the dendrogram via its default arguments. You can [read more here](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.dendrogram.html#scipy.cluster.hierarchy.dendrogram). 
1. Along the x-axis of the dendrogram:
    1. numbers without parentheses are the index positions (row numbers) of invidual observations (row numbers)
    1. numbers appearing in parentheses indicate the number of observations in that branch
1. We used the single linkage criterion, which is what causes the "chaining" behavior that can be seen in this subsection of the dendrogram (notably on the right side of the graph where samples are merged into the cluster one by one)

Let's see what the fitted clusters actually look like. For this, we need to apply the `.fit_predict()` method to apply our cluster model to the `moons_X` data. This will give us the label of the cluster that each observation is assigned to. For example, the first observation is assigned to a cluster labeled `485`.Notice we have a lot of unique values i.e. a lot of unique resulting clusters.

In [ ]:
agg_cluster_single_predicted = AgglomerativeClustering(
    linkage = 'single', 
    distance_threshold=0, 
    n_clusters=None).fit_predict(moons_X)
agg_cluster_single_predicted

Let's plot these clusters where we set `c` color to be the cluster labels. Again, we see a lot of unique clusters.

In [ ]:
plt.scatter(moons_X[:,0], moons_X[:,1], c = agg_cluster_single_predicted) 
plt.show()

**Question 1**:

- **Part A**: Apply agglomerative clustering using ward linkage to the "moons" data set. Display the first 10 levels of the resulting dendrogram.
- **Part B**: How does this dendrogram compare to the example that used single linkage? Highlight any noticeable differences you see and what they say about the nature of the resulting clusters.
- **Part C**: Plot the observations' cluster labels. What do you notice different?

### Cluster Labels

It is possible to "cut" a hierarchical clustering model to obtain a set $k$ non-overlapping clusters.

This practice allows us to compare the performance of hierarchical methods to alternatives like $k$-means, at least when we know the true clusters like we do in the simulated "moons" and "blobs" examples. In real-world applications, we must use our own judgement to decide if a clustering algorithm is producing useful and appropriate results.

The number of clusters can be chosen based upon context of the application, or inspection of the dendrogram to find a location where the height differences between merges begins to become large (an approach analogous to the "elbow" approach in 
$k$-means).

Below we refit our single linkage agglomerative clustering using $k=3$ and display the results. In order to plot the results, we use the `fit_predict()` method to get the predicted cluster labels for the given data:

In [ ]:
agg_cluster_single_3 = AgglomerativeClustering(
    linkage = 'single', 
    n_clusters=3).fit_predict(moons_X)
plt.scatter(moons_X[:,0], moons_X[:,1], c = agg_cluster_single_3) 
plt.show()

In this example we see the failure of the single linkage method. Other linkage criteria, such as 'complete' tend to perform a little better but not great:

In [ ]:
agg_cl_complete_3 = AgglomerativeClustering(
    linkage = 'complete', 
    n_clusters=3).fit_predict(moons_X)
plt.scatter(moons_X[:,0], moons_X[:,1], c = agg_cl_complete_3) 
plt.show()

You might also have noticed that we did not standardize our features, so these clusters might be disproportionately influenced by our first feature (shown on the x-axis). Unfortunately, standardization isn't enough to yield the results we desire:

In [ ]:
from sklearn.preprocessing import StandardScaler
moons_XS = StandardScaler().fit_transform(moons_X) 

agg_cl_complete_3 = AgglomerativeClustering(
    linkage = 'complete', 
    n_clusters=3).fit_predict(moons_XS)
plt.scatter(moons_XS[:,0], moons_XS[:,1], c = agg_cl_complete_3) 
plt.show()

**Question 2**:

- **Part A**: Analyze the "blobs" data set using agglomerative clustering to obtain $k=3$ clusters. Use scatter plots and your own judgements to decide upon an acceptable linkage criterion. Display a scatter plot showing cluster label assignments of your choice. Standardize the data before you begin your analysis.
- **Part B**: Analyze the "blobs" data set using $k$-means clustering to obtain $k=3$ clusters. Display a scatter plot showing cluster label assignments of your choice. Standardize the data before you begin your analysis.
- **Part C**: Comment on any differences. Which method to you prefer and why?